# 高速里程点与 PeMS 站点对比可视化 V2

## 功能
1. 支持**多个里程点文件**（如同时加载 99N 和 99S）
2. 支持**多种底图**（OSM、Google Maps、Esri 卫星图等）
3. 绘制 District 3 区域内的 PeMS 站点
4. 对比观察偏移情况

In [1]:
import pandas as pd
import numpy as np
import os
import glob
from math import radians, sin, cos, sqrt, atan2
import folium
from folium import plugins
import warnings
warnings.filterwarnings('ignore')

# ============== 配置 ==============

# 多个里程点文件配置
# 格式: {"标签名": "文件路径"}
POSTMILE_FILES = {
    "99N": "./99N.xlsx",
    "99S": "./99S.xlsx",
}

# 里程点颜色配置（不同方向用不同颜色）
POSTMILE_COLORS = {
    "99N": "#1976D2",    # 蓝色
    "99S": "#E91E63",    # 粉红色
    "default": "#607D8B", # 灰色
}

# PeMS 元数据目录
META_DIR = "../d03_meta"

# 目标高速和方向（用于筛选 PeMS 站点）
TARGET_FWY = "99"
TARGET_DIRS = ["N", "S"]  # 支持多个方向

# 输出目录
OUTPUT_DIR = "./output/postmile_comparison_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("配置完成！")
print(f"里程点文件: {list(POSTMILE_FILES.keys())}")
print(f"目标高速: {TARGET_FWY}")
print(f"目标方向: {TARGET_DIRS}")

配置完成！
里程点文件: ['99N', '99S']
目标高速: 99
目标方向: ['N', 'S']


## 1. 加载多个里程点文件

In [2]:
# 加载所有里程点文件
postmile_data = {}

for label, filepath in POSTMILE_FILES.items():
    if os.path.exists(filepath):
        df = pd.read_excel(filepath)
        df['Direction'] = label  # 添加方向标签
        postmile_data[label] = df
        print(f"\n{label}:")
        print(f"  文件: {filepath}")
        print(f"  记录数: {len(df)}")
        print(f"  列名: {list(df.columns)}")
        print(f"  Abs PM 范围: {df['Abs PM'].min():.2f} - {df['Abs PM'].max():.2f}")
    else:
        print(f"\n警告: 文件不存在 - {filepath}")

print(f"\n成功加载 {len(postmile_data)} 个里程点文件")


99N:
  文件: ./99N.xlsx
  记录数: 4163
  列名: ['District', 'County', 'CA PM', 'Abs PM', 'Length', 'Latitude', 'Longitude', 'Direction']
  Abs PM 范围: 0.00 - 416.20

99S:
  文件: ./99S.xlsx
  记录数: 4162
  列名: ['District', 'County', 'CA PM', 'Abs PM', 'Length', 'Latitude', 'Longitude', 'Direction']
  Abs PM 范围: 0.00 - 416.10

成功加载 2 个里程点文件


In [3]:
# 合并所有里程点数据
if postmile_data:
    all_postmiles = pd.concat(postmile_data.values(), ignore_index=True)
    print(f"合并后总记录数: {len(all_postmiles)}")
    print(f"\n各方向记录数:")
    print(all_postmiles['Direction'].value_counts())
else:
    all_postmiles = pd.DataFrame()
    print("警告: 没有加载任何里程点数据")

合并后总记录数: 8325

各方向记录数:
Direction
99N    4163
99S    4162
Name: count, dtype: int64


## 2. 加载 PeMS 站点元数据

In [4]:
# 查找 District 3 元数据文件
pattern = os.path.join(META_DIR, "d03_text_meta_*.txt")
meta_files = glob.glob(pattern)

if not meta_files:
    raise FileNotFoundError("未找到 D3 元数据文件")

meta_file = sorted(meta_files)[-1]
print(f"使用元数据文件: {meta_file}")

META_COLUMNS = [
    'ID', 'Fwy', 'Dir', 'District', 'County', 'City',
    'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
    'Type', 'Lanes', 'Name', 'User_ID_1', 'User_ID_2',
    'User_ID_3', 'User_ID_4'
]

meta_df = pd.read_csv(meta_file, sep='\t', names=META_COLUMNS, header=0, dtype={'ID': str, 'Fwy': str})
print(f"加载 {len(meta_df)} 条记录")

使用元数据文件: ../d03_meta/d03_text_meta_2025_12_30.txt
加载 1903 条记录


In [5]:
# 筛选目标高速站点
pems_stations = meta_df[
    (meta_df['Fwy'] == TARGET_FWY) & 
    (meta_df['Dir'].isin(TARGET_DIRS))
].copy()

# 有效坐标检查
pems_stations = pems_stations[
    (pems_stations['Latitude'].notna()) & 
    (pems_stations['Longitude'].notna()) &
    (pems_stations['Latitude'] > 30) &
    (pems_stations['Latitude'] < 42)
].copy()

pems_stations = pems_stations.sort_values(['Dir', 'Abs_PM']).reset_index(drop=True)

print(f"\n{TARGET_FWY} 高速站点 (D3 区域):")
print(f"  站点数: {len(pems_stations)}")
if len(pems_stations) > 0:
    print(f"\n方向分布:")
    print(pems_stations['Dir'].value_counts())
    print(f"\n类型分布:")
    print(pems_stations['Type'].value_counts())


99 高速站点 (D3 区域):
  站点数: 334

方向分布:
Dir
S    175
N    159
Name: count, dtype: int64

类型分布:
Type
ML    160
OR     70
HV     55
FR     46
FF      3
Name: count, dtype: int64


## 3. 多底图可视化

In [6]:
def create_multi_layer_map(postmile_data, pems_df, postmile_colors, output_path):
    """
    创建支持多底图、多里程点文件的对比地图
    
    参数:
        postmile_data: Dict[label -> DataFrame] 里程点数据
        pems_df: PeMS 站点 DataFrame
        postmile_colors: Dict[label -> color] 里程点颜色
        output_path: 输出文件路径
    """
    # 站点颜色（按类型）
    station_colors = {
        'ML': '#E53935',    # 红色
        'HV': '#8E24AA',    # 紫色
        'OR': '#43A047',    # 绿色
        'FR': '#FB8C00',    # 橙色
        'FF': '#FDD835',    # 黄色
    }
    
    # 计算地图中心
    all_lats = []
    all_lons = []
    for df in postmile_data.values():
        all_lats.extend(df['Latitude'].tolist())
        all_lons.extend(df['Longitude'].tolist())
    if len(pems_df) > 0:
        all_lats.extend(pems_df['Latitude'].tolist())
        all_lons.extend(pems_df['Longitude'].tolist())
    
    center_lat = np.mean(all_lats) if all_lats else 38.5
    center_lon = np.mean(all_lons) if all_lons else -121.5
    
    # 创建地图（默认无底图，通过图层控制选择）
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=10,
        tiles=None  # 不设置默认底图
    )
    
    # ========== 添加多种底图 ==========
    
    # 1. OpenStreetMap
    folium.TileLayer(
        tiles='OpenStreetMap',
        name='OpenStreetMap',
        control=True
    ).add_to(m)
    
    # 2. Google Maps 街道图
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',
        attr='Google Maps',
        name='Google Maps 街道',
        control=True
    ).add_to(m)
    
    # 3. Google Maps 卫星图
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
        attr='Google Satellite',
        name='Google 卫星图',
        control=True
    ).add_to(m)
    
    # 4. Google Maps 混合图（卫星+标签）
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
        attr='Google Hybrid',
        name='Google 混合图',
        control=True
    ).add_to(m)
    
    # 5. Esri 卫星图
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri World Imagery',
        name='Esri 卫星图',
        control=True
    ).add_to(m)
    
    # 6. CartoDB Positron（浅色简洁）
    folium.TileLayer(
        tiles='CartoDB positron',
        name='CartoDB 浅色',
        control=True
    ).add_to(m)
    
    # 7. CartoDB Dark Matter（深色）
    folium.TileLayer(
        tiles='CartoDB dark_matter',
        name='CartoDB 深色',
        control=True
    ).add_to(m)
    
    # ========== 添加里程点图层 ==========
    for label, df in postmile_data.items():
        color = postmile_colors.get(label, postmile_colors.get('default', '#1976D2'))
        
        postmile_group = folium.FeatureGroup(name=f'里程点 {label}')
        
        for _, row in df.iterrows():
            folium.CircleMarker(
                [row['Latitude'], row['Longitude']],
                radius=4,
                color=color,
                fill=True,
                fillColor=color,
                fillOpacity=0.8,
                weight=1,
                popup=f"{label}<br>PM: {row['Abs PM']:.2f}<br>County: {row['County']}",
                tooltip=f"{label} PM {row['Abs PM']:.1f}"
            ).add_to(postmile_group)
        
        postmile_group.add_to(m)
    
    # ========== 添加 PeMS 站点图层（按方向分组）==========
    if len(pems_df) > 0:
        for direction in pems_df['Dir'].unique():
            dir_df = pems_df[pems_df['Dir'] == direction]
            pems_group = folium.FeatureGroup(name=f'PeMS {TARGET_FWY}{direction}')
            
            for _, row in dir_df.iterrows():
                color = station_colors.get(row['Type'], '#888888')
                
                folium.CircleMarker(
                    [row['Latitude'], row['Longitude']],
                    radius=8,
                    color='white',
                    weight=2,
                    fill=True,
                    fillColor=color,
                    fillOpacity=0.9,
                    popup=folium.Popup(
                        f"<b>{row['ID']}</b><br>"
                        f"Type: {row['Type']}<br>"
                        f"Dir: {row['Dir']}<br>"
                        f"Abs_PM: {row['Abs_PM']:.3f}<br>"
                        f"County: {row['County']}<br>"
                        f"Name: {row['Name']}<br>"
                        f"Lat: {row['Latitude']:.6f}<br>"
                        f"Lon: {row['Longitude']:.6f}",
                        max_width=300
                    ),
                    tooltip=f"{row['ID']} ({row['Type']}) {row['Dir']} PM={row['Abs_PM']:.2f}"
                ).add_to(pems_group)
            
            pems_group.add_to(m)
    
    # 添加图层控制
    folium.LayerControl(collapsed=False).add_to(m)
    
    # 生成动态图例
    postmile_legend_items = ''.join([
        f'<div><span style="display:inline-block; width:12px; height:12px; background:{postmile_colors.get(label, "#1976D2")}; border-radius:50%; margin-right:5px;"></span>{label}</div>'
        for label in postmile_data.keys()
    ])
    
    legend_html = f"""
    <div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; 
                background-color: white; padding: 15px; border: 2px solid #333;
                border-radius: 8px; font-size: 12px; font-family: Arial;
                max-height: 400px; overflow-y: auto;">
        <div style="font-weight: bold; margin-bottom: 10px; font-size: 14px;">图例</div>
        
        <div style="font-weight: bold; margin-bottom: 5px;">Caltrans 里程点:</div>
        {postmile_legend_items}
        
        <div style="font-weight: bold; margin-top: 10px; margin-bottom: 5px;">PeMS 站点:</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:#E53935; border-radius:50%; margin-right:5px;"></span>ML (主线)</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:#8E24AA; border-radius:50%; margin-right:5px;"></span>HV (HOV)</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:#43A047; border-radius:50%; margin-right:5px;"></span>OR (入口)</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:#FB8C00; border-radius:50%; margin-right:5px;"></span>FR (出口)</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:#FDD835; border-radius:50%; margin-right:5px;"></span>FF (连接器)</div>
        
        <div style="margin-top: 10px; font-size: 10px; color: #666;">
            使用右上角图层控制<br>切换底图和数据图层
        </div>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    
    m.save(output_path)
    print(f"地图已保存: {output_path}")
    return m


print("可视化函数定义完成")

可视化函数定义完成


In [7]:
# 创建多底图对比地图
if postmile_data:
    comparison_map = create_multi_layer_map(
        postmile_data,
        pems_stations,
        POSTMILE_COLORS,
        os.path.join(OUTPUT_DIR, 'postmile_pems_multi_basemap.html')
    )
    comparison_map
else:
    print("错误: 没有里程点数据可显示")

地图已保存: ./output/postmile_comparison_v2/postmile_pems_multi_basemap.html


## 4. 偏移分析

In [8]:
def calc_distance_miles(lat1, lon1, lat2, lon2):
    """Haversine 距离（英里）"""
    R = 3958.8
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c


def find_nearest_postmile(pems_station, postmile_df):
    """
    找到距离 PeMS 站点最近的里程点
    """
    pems_pm = pems_station['Abs_PM']
    pems_lat = pems_station['Latitude']
    pems_lon = pems_station['Longitude']
    pems_dir = pems_station['Dir']
    
    # 1. 按 PM 匹配
    pm_diff = abs(postmile_df['Abs PM'] - pems_pm)
    pm_match_idx = pm_diff.idxmin()
    pm_match = postmile_df.loc[pm_match_idx]
    pm_match_dist = calc_distance_miles(pems_lat, pems_lon, pm_match['Latitude'], pm_match['Longitude'])
    
    # 2. 按坐标匹配
    distances = postmile_df.apply(
        lambda row: calc_distance_miles(pems_lat, pems_lon, row['Latitude'], row['Longitude']),
        axis=1
    )
    coord_match_idx = distances.idxmin()
    coord_match = postmile_df.loc[coord_match_idx]
    coord_match_dist = distances[coord_match_idx]
    
    return {
        'pems_id': pems_station['ID'],
        'pems_type': pems_station['Type'],
        'pems_dir': pems_dir,
        'pems_pm': pems_pm,
        'pems_lat': pems_lat,
        'pems_lon': pems_lon,
        'pm_match_pm': pm_match['Abs PM'],
        'pm_match_dist_mi': pm_match_dist,
        'coord_match_pm': coord_match['Abs PM'],
        'coord_match_dist_mi': coord_match_dist,
        'pm_difference': coord_match['Abs PM'] - pems_pm,
    }


print("分析函数定义完成")

分析函数定义完成


In [9]:
# 对每个方向分别分析偏移
all_analysis = []

for direction in TARGET_DIRS:
    # 获取该方向的里程点和站点
    dir_label = f"{TARGET_FWY}{direction}"
    
    if dir_label not in postmile_data:
        print(f"跳过 {dir_label}: 无里程点数据")
        continue
    
    dir_postmiles = postmile_data[dir_label]
    dir_stations = pems_stations[pems_stations['Dir'] == direction]
    
    if len(dir_stations) == 0:
        print(f"跳过 {dir_label}: 无 PeMS 站点")
        continue
    
    print(f"\n分析 {dir_label}:")
    print(f"  里程点数: {len(dir_postmiles)}")
    print(f"  站点数: {len(dir_stations)}")
    
    for _, station in dir_stations.iterrows():
        result = find_nearest_postmile(station, dir_postmiles)
        result['postmile_source'] = dir_label
        all_analysis.append(result)

if all_analysis:
    analysis_df = pd.DataFrame(all_analysis)
    print(f"\n总共分析 {len(analysis_df)} 个站点")
else:
    analysis_df = pd.DataFrame()
    print("\n没有可分析的数据")


分析 99N:
  里程点数: 4163
  站点数: 159

分析 99S:
  里程点数: 4162
  站点数: 175

总共分析 334 个站点


In [10]:
# 显示偏移分析结果
if len(analysis_df) > 0:
    print("=" * 90)
    print("偏移分析详情")
    print("=" * 90)
    print(f"{'ID':<10} {'Type':<4} {'Dir':<3} {'PeMS_PM':>8} {'PM匹配距离':>12} {'坐标匹配PM':>10} {'坐标匹配距离':>12} {'PM差异':>10}")
    print("-" * 90)
    
    for _, row in analysis_df.iterrows():
        print(f"{row['pems_id']:<10} {row['pems_type']:<4} {row['pems_dir']:<3} {row['pems_pm']:>8.2f} "
              f"{row['pm_match_dist_mi']*5280:>10.0f} ft {row['coord_match_pm']:>10.2f} "
              f"{row['coord_match_dist_mi']*5280:>10.0f} ft {row['pm_difference']:>+10.2f}")

偏移分析详情
ID         Type Dir  PeMS_PM       PM匹配距离     坐标匹配PM       坐标匹配距离       PM差异
------------------------------------------------------------------------------------------
3414051    ML   N     274.66        210 ft     274.70        210 ft      +0.04
3414056    FR   N     274.74        215 ft     274.70        215 ft      -0.04
319102     FR   N     275.13        163 ft     275.10        163 ft      -0.03
319091     ML   N     275.40          0 ft     275.40          0 ft      +0.00
319092     OR   N     275.40          0 ft     275.40          0 ft      +0.00
3414066    OR   N     278.13        186 ft     278.10        186 ft      -0.03
3414064    ML   N     278.24        229 ft     278.20        229 ft      -0.04
3027041    ML   N     281.67        164 ft     281.70        164 ft      +0.03
3027042    FR   N     281.67        164 ft     281.70        164 ft      +0.03
3027012    OR   N     281.75        264 ft     281.80        264 ft      +0.05
3027011    ML   N     281.76       

In [11]:
# 按方向统计偏移
if len(analysis_df) > 0:
    print("\n" + "=" * 60)
    print("各方向偏移统计")
    print("=" * 60)
    
    for direction in analysis_df['pems_dir'].unique():
        dir_data = analysis_df[analysis_df['pems_dir'] == direction]
        
        print(f"\n{TARGET_FWY}{direction}:")
        print(f"  站点数: {len(dir_data)}")
        print(f"  PM匹配偏移 - 平均: {dir_data['pm_match_dist_mi'].mean()*5280:.0f} ft, "
              f"最大: {dir_data['pm_match_dist_mi'].max()*5280:.0f} ft")
        print(f"  坐标匹配偏移 - 平均: {dir_data['coord_match_dist_mi'].mean()*5280:.0f} ft, "
              f"最大: {dir_data['coord_match_dist_mi'].max()*5280:.0f} ft")
        print(f"  PM差异 - 平均: {dir_data['pm_difference'].mean():+.3f} mi, "
              f"范围: [{dir_data['pm_difference'].min():+.3f}, {dir_data['pm_difference'].max():+.3f}]")


各方向偏移统计

99N:
  站点数: 159
  PM匹配偏移 - 平均: 134 ft, 最大: 276 ft
  坐标匹配偏移 - 平均: 134 ft, 最大: 276 ft
  PM差异 - 平均: +0.007 mi, 范围: [-0.046, +0.050]

99S:
  站点数: 175
  PM匹配偏移 - 平均: 131 ft, 最大: 375 ft
  坐标匹配偏移 - 平均: 131 ft, 最大: 375 ft
  PM差异 - 平均: +0.003 mi, 范围: [-0.049, +0.050]


In [12]:
# 保存分析结果
if len(analysis_df) > 0:
    output_file = os.path.join(OUTPUT_DIR, 'offset_analysis.csv')
    analysis_df.to_csv(output_file, index=False)
    print(f"\n分析结果已保存: {output_file}")


分析结果已保存: ./output/postmile_comparison_v2/offset_analysis.csv


## 5. 带偏移连线的地图

In [13]:
def create_offset_map(analysis_df, postmile_data, postmile_colors, output_path):
    """
    创建显示偏移连线的地图
    """
    if len(analysis_df) == 0:
        print("没有分析数据")
        return None
    
    station_colors = {
        'ML': '#E53935',
        'HV': '#8E24AA',
        'OR': '#43A047',
        'FR': '#FB8C00',
        'FF': '#FDD835',
    }
    
    center_lat = analysis_df['pems_lat'].mean()
    center_lon = analysis_df['pems_lon'].mean()
    
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=10,
        tiles=None
    )
    
    # 添加底图
    folium.TileLayer('OpenStreetMap', name='OpenStreetMap').add_to(m)
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
        attr='Google Hybrid',
        name='Google 混合图'
    ).add_to(m)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Esri 卫星图'
    ).add_to(m)
    
    # 添加里程点连线
    for label, df in postmile_data.items():
        color = postmile_colors.get(label, '#1976D2')
        sorted_df = df.sort_values('Abs PM')
        coords = [[row['Latitude'], row['Longitude']] for _, row in sorted_df.iterrows()]
        
        folium.PolyLine(
            coords,
            color=color,
            weight=3,
            opacity=0.8,
            name=f'里程线 {label}'
        ).add_to(m)
    
    # 添加站点和偏移连线
    for _, row in analysis_df.iterrows():
        color = station_colors.get(row['pems_type'], '#888888')
        pm_dist_ft = row['pm_match_dist_mi'] * 5280
        
        # PeMS 站点
        folium.CircleMarker(
            [row['pems_lat'], row['pems_lon']],
            radius=10,
            color='white',
            weight=2,
            fill=True,
            fillColor=color,
            fillOpacity=0.9,
            popup=f"{row['pems_id']} ({row['pems_type']})<br>PM偏移: {pm_dist_ft:.0f} ft",
            tooltip=f"{row['pems_id']} 偏移{pm_dist_ft:.0f}ft"
        ).add_to(m)
        
        # 偏移连线
        if pm_dist_ft > 50:
            # 找到对应的里程点坐标
            source_label = row['postmile_source']
            if source_label in postmile_data:
                pm_df = postmile_data[source_label]
                pm_match = pm_df.iloc[(pm_df['Abs PM'] - row['pm_match_pm']).abs().argmin()]
                
                line_color = '#F44336' if pm_dist_ft > 500 else '#FFC107'
                folium.PolyLine(
                    [[row['pems_lat'], row['pems_lon']],
                     [pm_match['Latitude'], pm_match['Longitude']]],
                    color=line_color,
                    weight=2,
                    opacity=0.8,
                    dash_array='5,5',
                    tooltip=f"偏移: {pm_dist_ft:.0f} ft"
                ).add_to(m)
    
    folium.LayerControl().add_to(m)
    m.save(output_path)
    print(f"偏移地图已保存: {output_path}")
    return m


if len(analysis_df) > 0:
    offset_map = create_offset_map(
        analysis_df,
        postmile_data,
        POSTMILE_COLORS,
        os.path.join(OUTPUT_DIR, 'offset_visualization.html')
    )
    offset_map

偏移地图已保存: ./output/postmile_comparison_v2/offset_visualization.html


## 总结

### 功能说明

1. **多里程点文件**: 在 `POSTMILE_FILES` 字典中配置多个文件
2. **多底图支持**: 
   - OpenStreetMap
   - Google Maps 街道/卫星/混合
   - Esri 卫星图
   - CartoDB 浅色/深色
3. **图层控制**: 可以单独开关每个方向的里程点和站点

### 使用方法

1. 修改 `POSTMILE_FILES` 添加你的里程点文件
2. 修改 `POSTMILE_COLORS` 设置各方向颜色
3. 修改 `TARGET_FWY` 和 `TARGET_DIRS` 筛选 PeMS 站点
4. 运行所有单元格生成地图